# Amazon Bedrock AgentCore에서 Microsoft Entra ID On-Behalf-Of(OBO) 사용

이 Notebook에서는 최종 사용자의 신원을 end-to-end로 유지하면서 전용 MCP server를 통해 downstream resource(Microsoft Graph)를 호출하는 Amazon Bedrock AgentCore Runtime 에이전트의 **On-Behalf-Of(OBO) token exchange pattern**을 살펴봅니다.


## 사용 사례

**에이전트가 원본 사용자 JWT를 서비스 경계 너머로 전달하지 않고 사용자를 대신해 해당 사용자의 SaaS 데이터(예: Microsoft Graph)를 읽는 개인 생산성 도우미입니다.**

*예: 임원 비서 copilot. Alex가 Microsoft 계정으로 로그인한 뒤 "who am I and what's on my profile?"이라고 묻습니다. 에이전트는 공유 service account가 아니라 Alex로 Microsoft Graph를 호출하므로 다른 사람이 아닌 Alex의 프로필이 반환됩니다. 에이전트는 AgentCore Identity에 Alex의 inbound JWT를 Graph scope로 제한된 delegation token으로 교환하도록 요청하고, 해당 토큰을 MCP 도구에 전달해 Graph를 호출합니다. Bob이 같은 prompt를 실행하면 Bob의 프로필이 반환됩니다.*

**이 pattern이 적합한 경우:**
- AgentCore Runtime에 자체 MCP server를 구축했거나 구축할 예정입니다.
- MCP server가 사용자 위임 OAuth 2.0 토큰을 허용하는 서드 파티 API(Microsoft Graph, Salesforce, Google 등)를 호출합니다.
- 대화 도중 authorization URL을 표시하는 대신 사용자가 로그인할 때 한 번만 동의하도록 구성하려고 합니다.
- downstream 호출에 사용자 신원과 에이전트 신원을 모두 포함하여 resource server에서 "agent A acting on behalf of user U"를 감사할 수 있어야 합니다.

**이 pattern이 적합하지 않은 경우:**
- MCP server가 회사 소유의 내부 시스템을 호출하며 도구 계층에서 사용자 신원이 필요하지 않습니다. 이 경우 순수 M2M이 더 간단합니다.
- 도구가 응답에 포함된 authorization URL을 통해 직접 user consent를 처리해야 합니다. 이 경우 USER_FEDERATION을 사용하는 3LO가 더 적합합니다.
- MCP server를 작성하지 않고 기존 OpenAPI를 도구로 노출하려고 합니다. AgentCore Gateway를 사용하세요.
- IdP가 OAuth 2.0 token exchange를 지원하지 않습니다. 아키텍처 섹션의 호환성 표를 참조하세요.


## Microsoft Entra ID 개요

Microsoft Entra ID(이전 Azure Active Directory)는 Microsoft의 클라우드 기반 ID 및 액세스 관리 서비스입니다. Microsoft 365, Azure 및 수천 개의 기타 SaaS 애플리케이션을 위한 중앙 IdP 역할을 합니다.

**주요 기능:**
- **Single Sign-On(SSO)**: 사용자는 한 번 인증하여 여러 애플리케이션에 액세스할 수 있습니다.
- **OAuth 2.0 On-Behalf-Of flow**: middle-tier 서비스가 inbound 사용자 토큰을 downstream 토큰으로 교환하여 서비스 hop 전반에서 사용자 신원을 유지할 수 있습니다.
- **애플리케이션 통합**: OAuth 2.0, OpenID Connect, SAML을 지원합니다.

**참고:** Microsoft Entra ID는 AWS 서비스가 아닙니다. 비용과 라이선스는 [Microsoft Entra ID 문서](https://learn.microsoft.com/en-us/entra/)를 참조하세요.


## 학습 목표

1. **client_credentials(M2M)**, **USER_FEDERATION(3LO)**, **On-Behalf-Of(OBO)** pattern의 차이와 각각을 선택할 시점을 이해합니다.
2. Agent app(middle-tier)과 MCP Server app(protected resource), 두 개의 Entra ID app registration을 설정합니다.
3. OBO token exchange용 AgentCore Identity OAuth2 credential provider를 생성합니다.
4. 에이전트 내부에서 `GetResourceOauth2Token(ON_BEHALF_OF_TOKEN_EXCHANGE)`을 사용하여 inbound 사용자 JWT를 Graph scope로 제한된 delegation token으로 교환하고, 사용자 JWT 자체가 agent-to-MCP 경계를 넘지 않도록 합니다.
5. 사용자 인증 JWT로 에이전트를 호출하고 delegation 흐름을 end-to-end로 관찰합니다.


## 튜토리얼 세부 정보

| 항목 | 세부 정보 |
|:------------|:--------|
| 튜토리얼 유형 | 단계별 |
| 구성 요소 | AgentCore Runtime의 에이전트 + AgentCore Runtime의 MCP Server + Microsoft Graph |
| Agentic Framework | Strands Agents |
| LLM model | Anthropic Claude Sonnet 4.5 |
| 예제 난이도 | 중간 |
| 사용 SDK | Amazon Bedrock AgentCore SDK, Bedrock AgentCore Starter Toolkit, boto3, MSAL |
| Inbound Auth | User JWT via Entra ID Device Code Flow |
| Outbound Auth (Agent→MCP) | Entra ID M2M (client_credentials) |
| Outbound Auth (Agent→Graph) | **Entra ID On-Behalf-Of (OBO)** via AgentCore Identity |


## 아키텍처

아래 다이어그램은 사용자의 prompt가 사용자를 대신한 Microsoft Graph 호출로 이어지는 과정을 보여 줍니다. 다이어그램의 각 번호는 이어지는 흐름 섹션에서 설명합니다.

<div style="text-align:center">
    <img src="images/architecture.png" width="95%" alt="Architecture: User signs in with Microsoft Entra ID and receives a user JWT. The agent on AgentCore Runtime exchanges that JWT for a Microsoft Graph delegation token via AgentCore Identity, then calls an MCP server on a second AgentCore Runtime, passing the delegation token in a custom request header. The MCP server uses that token as the Bearer credential when calling Microsoft Graph."/>
</div>

### 흐름

1. 사용자가 Microsoft Entra ID(device code flow)로 인증하고 `aud = <Agent app client_id>`인 사용자 JWT를 받습니다.
2. 사용자가 `Authorization: Bearer <user_jwt>`로 에이전트를 호출합니다. AgentCore Runtime은 Agent app의 `customJWTAuthorizer`를 기준으로 JWT를 검증하고 요청을 에이전트에 전달합니다.
3. 에이전트가 사용자의 prompt로 Amazon Bedrock의 LLM을 호출하면 LLM이 도구 호출의 필요 여부를 판단합니다.
4. 에이전트가 AgentCore Identity의 `GetResourceOauth2Token(oauth2Flow=ON_BEHALF_OF_TOKEN_EXCHANGE)`을 호출합니다. AgentCore Identity는 Agent app의 client 자격 증명을 사용해 Entra와 OBO 교환을 중개하고, 에이전트를 acting service로 하여 사용자를 나타내는 Graph scope의 delegation token을 반환합니다.
5. 에이전트가 두 HTTP header와 함께 MCP server를 호출합니다. `Authorization: Bearer <m2m_token>`은 MCP server의 JWT authorizer에 에이전트를 식별하고, `X-Amzn-Bedrock-AgentCore-Runtime-Custom-Graph-Token: <graph_obo_token>`은 사용자 delegation을 전달합니다. 역할 분리 관계는 아래 **보안 고려 사항**을 참조하세요. 두 토큰 모두 LLM context에 들어가지 않습니다.
6. MCP 도구가 request context에서 Graph OBO token을 읽고 이를 Bearer 자격 증명으로 사용해 Microsoft Graph를 호출합니다. Graph는 토큰을 검증하고 사용자 신원을 확인한 뒤 사용자의 데이터를 반환합니다.
7. 응답은 MCP → Agent → User 순서로 돌아갑니다.

### 더 단순한 pattern 대신 OBO를 사용하는 이유

| Pattern | Agent → MCP 경계를 넘는 정보 | 사용자 신원 유지 여부 | 동의 UX |
|---|---|---|---|
| 순수 M2M | 에이전트의 M2M token만 | 아니요, 에이전트가 자신으로 작업 | 요청별 동의 없음(admin consent 한 번) |
| 사용자 JWT 전달 | M2M token + inbound 사용자 JWT | 예, 하지만 토큰 `aud`가 MCP와 일치하지 않음 | 도구별 authorization URL |
| OBO(이 샘플) | M2M token + Graph scope의 delegation token | 예, 에이전트가 사용자를 대신한 것으로 기록 | 로그인 시 한 번 동의 |

### IdP 호환성

이 OBO pattern에는 OAuth 2.0 token exchange를 지원하는 IdP가 필요합니다. 작성 시점의 지원 현황은 다음과 같습니다.

| IdP | OBO 지원 | AgentCore provider |
|---|---|---|
| Microsoft Entra ID | Microsoft OBO flow로 지원 | `MicrosoftOauth2`(사전 구성) |
| Okta | OAuth 2.0 Token Exchange로 지원 | `OktaOauth2` 또는 `CustomOauth2` |
| Auth0 | Custom Token Exchange로 지원 | `CustomOauth2` |
| Google OAuth | OBO 미지원, domain-wide delegation 또는 Google STS 사용 | OBO 해당 없음 |
| Amazon Cognito | OBO 미지원, IAM Identity Center Trusted Identity Propagation 사용 | OBO 해당 없음 |
| GitHub / Slack / Salesforce / Atlassian / LinkedIn / X | 현재 AgentCore에서 OBO 연결 미지원 | 대신 3LO 사용 |

각 플랫폼에서 token exchange가 작동하는 방식은 해당 IdP 문서를 참조하세요.

현재 provider 목록과 grant type 옵션은 [AgentCore OBO 개발자 가이드](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/on-behalf-of-token-exchange.html)를 참조하세요.


## 사전 요구 사항

### Entra ID 설정

이 샘플에서는 trust 아키텍처에서 각각 고유한 역할을 담당하는 **두 개의 Entra app registration**을 사용합니다.

**`AgentCore - Agent`**는 middle-tier 서비스입니다.

- 사용자는 이 앱에 로그인합니다. inbound 사용자 JWT의 값은 `aud = <AgentCore-Agent client_id>`입니다.
- 이 앱이 OBO token exchange를 수행합니다. Entra는 이 앱의 client 자격 증명을 기반으로 Graph delegation token을 발급합니다.
- 이 앱이 MCP server를 호출합니다. MCP에 제시되는 M2M token의 audience는 MCP Server app의 client_id입니다.

하나의 앱이 세 가지 역할을 담당합니다. Microsoft OBO 문서는 이러한 역할을 하나의 앱으로 통합하는 방식을 명시적으로 권장합니다([*"Use of a single application"*](https://learn.microsoft.com/en-us/entra/identity-platform/v2-oauth2-on-behalf-of-flow#use-of-a-single-application) 참조). 로그인 역할과 middle-tier 역할을 분리하면 inbound JWT의 `aud`가 middle-tier app의 client_id와 일치해야 한다는 OBO 요구 사항을 위반하여 `AADSTS500131` 오류가 발생합니다.

**`AgentCore - MCP Server`**는 protected resource입니다.

- Application ID URI는 MCP server의 `customJWTAuthorizer`가 검증하는 **audience** 값입니다.
- `mcp_invoke` app role은 MCP 호출을 허용할 요청자를 결정하는 **권한 부여 경계**입니다.
- 자체 사용자, secret 또는 API 권한은 없습니다.

단계별 근거와 일반적인 `AADSTS*` 오류의 문제 해결 표가 포함된 **상세 포털 안내**는 **[ENTRA_SETUP.md](./ENTRA_SETUP.md)**를 참조하세요. 아래 체크리스트는 요약본입니다.

**`AgentCore - Agent` app에서:**

- **Authentication**: **Mobile and desktop applications** 플랫폼을 추가하고 `https://login.microsoftonline.com/common/oauth2/nativeclient`를 선택한 다음 **Allow public client flows**를 **Yes**로 설정합니다.
- **Certificates & secrets**: client secret을 생성합니다. 즉시 Value를 복사하세요. 이 값이 `ENTRA_AGENT_CLIENT_SECRET`입니다.
- **Expose an API**: Application ID URI를 `api://<client-id>`로 설정한 다음 `user_delegation`이라는 scope를 추가합니다(Admins and users can consent, State = Enabled).
- **API permissions**: Microsoft Graph **Delegated** 권한 `User.Read`를 추가한 다음 **Grant admin consent**를 클릭합니다.

**`AgentCore - MCP Server` app에서:**

- **Expose an API**: Application ID URI를 `api://<client-id>`로 설정합니다. scope는 추가하지 않습니다.
- **App roles**: Value가 `mcp_invoke`, Display name이 `Invoke MCP Server`, Allowed member types가 **Applications**, 상태가 Enabled인 app role을 생성합니다.

**다시 `AgentCore - Agent` app에서:**

- **API permissions**: **APIs my organization uses**에서 권한을 추가하고 `AgentCore - MCP Server`를 선택합니다. **Application permissions**를 선택한 후 `mcp_invoke`를 체크하고 **Grant admin consent**를 클릭합니다.

### 다음 값 수집

| 변수 | 출처 |
|----------|--------|
| `ENTRA_TENANT_ID` | Entra admin center → Overview → Tenant ID |
| `ENTRA_AGENT_CLIENT_ID` | AgentCore - Agent → Overview → Application (client) ID |
| `ENTRA_AGENT_CLIENT_SECRET` | 생성 시 복사한 secret 값 |
| `ENTRA_MCP_CLIENT_ID` | AgentCore - MCP Server → Overview → Application (client) ID |


In [ ]:
!pip install -r requirements.txt -q

## 1단계: AWS Session 및 import 설정


In [ ]:
import uuid
import urllib.parse
import os
import boto3
from boto3.session import Session


boto_session = Session()
sts = boto3.client("sts")
account_id = sts.get_caller_identity().get("Account")
region = boto_session.region_name or "us-west-2"
print(f"Account: {account_id}, Region: {region}")

## 2단계: 환경 변수 구성

placeholder 값을 실제 Entra ID app registration 정보로 바꾸세요.


In [ ]:
# Entra ID 테넌트
os.environ["ENTRA_TENANT_ID"] = "your-tenant-id"  # 교체

# Agent app - 사용자가 로그인하고 OBO를 수행하며 M2M으로 MCP server에 인증하는 단일 앱.
# secret이 있는 confidential client이며 device code flow 로그인이 작동하도록
# "Allow public client flows"도 활성화되어 있음.
os.environ["ENTRA_AGENT_CLIENT_ID"] = "your-agent-app-client-id"  # 교체
os.environ["ENTRA_AGENT_CLIENT_SECRET"] = "your-agent-app-client-secret"  # 교체

# MCP Server app - 에이전트가 MCP server를 호출할 때 사용하는 M2M token의 audience가
# 이 앱의 Application ID URI임. secret을 보유하지 않으며 MCP server의 authorizer가
# 검증하는 mcp_invoke app role을 노출함.
os.environ["ENTRA_MCP_CLIENT_ID"] = "your-mcp-audience-client-id"  # 교체

## 3단계: MCP server 코드 생성

MCP server의 빌드 아티팩트(`Dockerfile`, `.bedrock_agentcore.yaml`)와 에이전트 아티팩트를 분리하기 위해 각각 자체 하위 디렉터리에서 배포합니다. MCP server는 `mcp/`, 에이전트는 `agent/`를 사용합니다. toolkit이 컨테이너를 빌드할 때 `configure()`에서 올바른 종속성을 가져오도록 각 하위 디렉터리에 `requirements.txt`를 복사합니다.

MCP server는 애플리케이션 계층에서 인증 방식에 의존하지 않습니다. Token Vault, workload token 또는 authorization URL을 관리하지 않습니다. 에이전트가 custom request header로 Graph OBO token을 전달하고 MCP 도구가 request context에서 이를 읽으므로 도구 signature에는 인증 파라미터가 없습니다.

MCP transport는 에이전트의 M2M token을 검증하는 AgentCore의 `customJWTAuthorizer`로 보호됩니다. 따라서 권한이 없는 요청자는 MCP surface에 접근할 수 없습니다. 특정 호출에서 도구에 필요한 Graph token을 가져와 전달하는 책임은 에이전트에 있으며 MCP 도구 자체는 인증을 처리하지 않습니다.


In [ ]:
%%writefile mcp/mcp_server_obo.py
"""Microsoft Graph 도구를 제공하는 MCP Server입니다.

인증 모델:
- MCP 전송: MCP Server 앱의 M2M 토큰에 대해 AgentCore customJWTAuthorizer로 권한을 부여합니다.
- Graph 호출: 에이전트가 Graph 범위 OBO 토큰을 사용자 지정 요청 헤더
              (X-Amzn-Bedrock-AgentCore-Runtime-Custom-Graph-Token)로 전송합니다. MCP Server는
              요청 컨텍스트에서 토큰을 읽고 Microsoft Graph의 Bearer 자격 증명으로 사용합니다.

Graph 토큰을 도구 인자가 아닌 헤더로 전달하는 이유:
- LLM이 자격 증명을 보거나 처리해서는 안 됩니다(RFC 9700 §4.9, OWASP LLM06).
- 따라서 도구 시그니처에는 비즈니스 계층 매개변수만 포함됩니다.
- 헤더는 전송 중 TLS로 보호되며, 사용자 인바운드 JWT와 동일한 신뢰 경계 안에 있는
  에이전트와 MCP Server에서만 볼 수 있습니다.
"""
import httpx
from mcp.server.fastmcp import FastMCP
from typing import Dict, Any

mcp = FastMCP(host='0.0.0.0', stateless_http=True)

GRAPH_BASE = 'https://graph.microsoft.com/v1.0'
# AgentCore는 Runtime에서 명시적으로 allowlist에 추가되고 이름이 'Authorization'이거나
# 'X-Amzn-Bedrock-AgentCore-Runtime-Custom-' 접두사가 있는 request header만 전달함.
# allowlist는 4단계에서 MCP Runtime에 설정함. 참고:
# https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/runtime-header-allowlist.html
GRAPH_TOKEN_HEADER = 'x-amzn-bedrock-agentcore-runtime-custom-graph-token'


def _get_graph_token() -> str:
    """에이전트가 이 요청에 첨부한 Graph OBO 토큰을 읽습니다."""
    ctx = mcp.get_context()
    headers = dict(ctx.request_context.request.headers)
    token = headers.get(GRAPH_TOKEN_HEADER, '').strip()
    if not token:
        raise RuntimeError(
            f"Missing Graph OBO token. Expected request header '{GRAPH_TOKEN_HEADER}' "
            "to carry the delegation token. Check that the header is allowlisted on the "
            "MCP runtime's requestHeaderConfiguration."
        )
    return token


@mcp.tool()
async def get_my_profile() -> Dict[str, Any]:
    """Return the signed-in user's Microsoft Graph profile."""
    access_token = _get_graph_token()
    async with httpx.AsyncClient() as client:
        r = await client.get(f'{GRAPH_BASE}/me',
                             headers={'Authorization': f'Bearer {access_token}'})
    if r.status_code != 200:
        return {'error': f'Graph returned {r.status_code}', 'body': r.text}
    p = r.json()
    return {
        'displayName': p.get('displayName'),
        'email': p.get('mail') or p.get('userPrincipalName'),
        'jobTitle': p.get('jobTitle'),
        'id': p.get('id'),
    }


if __name__ == '__main__':
    mcp.run(transport='streamable-http')


## 4단계: AgentCore Runtime에 MCP server 배포

starter toolkit은 현재 작업 디렉터리에 `Dockerfile`과 `.bedrock_agentcore.yaml`을 생성합니다. MCP와 에이전트 아티팩트를 분리하기 위해 구성 및 시작 전에 `os.chdir()`로 `mcp/` 디렉터리로 이동합니다. 이후 셀이 계속 작동하도록 `finally` 블록에서 원래 디렉터리로 돌아갑니다.

여기서는 두 가지 Runtime 구성이 중요합니다.

**1. `authorizerConfiguration`이 MCP transport를 보호합니다.** `customJWTAuthorizer`는 Agent app의 M2M token을 검증하고 `mcp_invoke` app role이 포함되어 있는지 확인합니다. OBO는 앞서 에이전트에서 수행됩니다. 여기서 MCP의 역할은 권한이 없는 요청자를 거부하는 것입니다.

**2. `requestHeaderConfiguration`을 통해 Graph OBO token이 MCP 도구에 도달합니다.** AgentCore Runtime은 기본적으로 request header를 제거하고 Runtime allowlist에 있는 header만 전달합니다. 두 가지 규칙이 적용됩니다([문서](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/runtime-header-allowlist.html)). header 이름은 `Authorization`이거나 `X-Amzn-Bedrock-AgentCore-Runtime-Custom-`으로 시작해야 하며, Runtime의 `requestHeaderAllowlist`에도 나열되어야 합니다. allowlist 항목이 없으면 MCP 컨테이너에서 header가 사라지고 도구에서 "missing token" 오류가 발생하며 LLM은 Graph를 호출하지 못한 채 사과합니다. M2M transport token용 `Authorization`과 Graph delegation token용 `X-Amzn-Bedrock-AgentCore-Runtime-Custom-Graph-Token`을 allowlist에 추가합니다.


In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime

# 이후 셀에서 경로를 올바르게 해석하도록 Notebook의 루트 디렉터리 저장
NOTEBOOK_DIR = os.getcwd()

mcp_runtime = Runtime()
mcp_discovery_url = (
    f"https://login.microsoftonline.com/{os.environ['ENTRA_TENANT_ID']}/.well-known/openid-configuration"
)
mcp_audience = f"api://{os.environ['ENTRA_MCP_CLIENT_ID']}"

# 에이전트가 Graph OBO token 전달에 사용하는 header 이름. AgentCore에서 허용하려면
# 'X-Amzn-Bedrock-AgentCore-Runtime-Custom-'으로 시작해야 함
# (위 4단계 Markdown 참조).
GRAPH_TOKEN_HEADER = "X-Amzn-Bedrock-AgentCore-Runtime-Custom-Graph-Token"

try:
    os.chdir("mcp")  # toolkit이 cwd에 Dockerfile과 .bedrock_agentcore.yaml을 작성함
    mcp_config = mcp_runtime.configure(
        entrypoint="mcp_server_obo.py",
        auto_create_execution_role=True,
        auto_create_ecr=True,
        requirements_file="requirements.txt",
        region=region,
        agent_name="entra_obo_mcp",
        protocol="MCP",
        authorizer_configuration={
            "customJWTAuthorizer": {
                "discoveryUrl": mcp_discovery_url,
                "allowedAudience": [mcp_audience],
                "customClaims": [
                    {
                        "inboundTokenClaimName": "roles",
                        "inboundTokenClaimValueType": "STRING_ARRAY",
                        "authorizingClaimMatchValue": {
                            "claimMatchValue": {"matchValueString": "mcp_invoke"},
                            "claimMatchOperator": "CONTAINS",
                        },
                    }
                ],
            }
        },
        request_header_configuration={
            "requestHeaderAllowlist": [
                "Authorization",
                GRAPH_TOKEN_HEADER,
            ]
        },
    )
finally:
    os.chdir(NOTEBOOK_DIR)
print("MCP Runtime configured")

In [ ]:
# MCP server 배포
try:
    os.chdir("mcp")
    mcp_launch = (
        mcp_runtime.launch()
    )  # CodeBuild 사용(로컬 Docker 불필요). Docker/Finch/Podman이 있으면 local_build=True 사용
finally:
    os.chdir(NOTEBOOK_DIR)
print(f"MCP Server deployed: {mcp_launch.agent_id}")

## 5단계: Credential Provider 생성

Agent app의 clientId/secret을 사용하는 단일 `MicrosoftOauth2` credential provider를 생성합니다. Runtime에서 서로 다른 두 OAuth 흐름에 **동일한 provider**를 재사용합니다.

- `oauth2Flow=M2M` → client_credentials grant: 에이전트가 MCP transport 권한 부여용 M2M token을 가져올 때 사용합니다.
- `oauth2Flow=ON_BEHALF_OF_TOKEN_EXCHANGE` → OBO exchange: 에이전트가 사용자를 대신하여 Graph scope의 delegation token을 가져올 때 사용합니다.

`get_resource_oauth2_token`은 호출별 파라미터로 흐름을 받으며 provider 자체는 client 자격 증명과 vendor 구성만 저장하므로 하나의 provider가 두 흐름을 모두 처리할 수 있습니다.

`MicrosoftOauth2` vendor에는 OBO 지원이 미리 연결되어 있습니다. 기본으로 사용하는 grant type mode는 [AgentCore OBO 개발자 가이드](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/on-behalf-of-token-exchange.html)를 참조하세요.


In [ ]:
agentcore_control = boto3.client("bedrock-agentcore-control", region_name=region)

CREDENTIAL_PROVIDER_NAME = "entra-agent-provider"

try:
    cp_response = agentcore_control.create_oauth2_credential_provider(
        name=CREDENTIAL_PROVIDER_NAME,
        credentialProviderVendor="MicrosoftOauth2",
        oauth2ProviderConfigInput={
            "microsoftOauth2ProviderConfig": {
                "clientId": os.environ["ENTRA_AGENT_CLIENT_ID"],
                "clientSecret": os.environ["ENTRA_AGENT_CLIENT_SECRET"],
                "tenantId": os.environ["ENTRA_TENANT_ID"],
            }
        },
    )
    credential_provider_arn = cp_response["credentialProviderArn"]
    print(f"Created credential provider: {credential_provider_arn}")
except agentcore_control.exceptions.ConflictException:
    print("Credential provider already exists, reusing it")
    cp_response = agentcore_control.get_oauth2_credential_provider(name=CREDENTIAL_PROVIDER_NAME)
    credential_provider_arn = cp_response["credentialProviderArn"]
    print(f"Using existing: {credential_provider_arn}")

## 6단계: 에이전트 코드 생성

에이전트의 `Dockerfile`과 `.bedrock_agentcore.yaml`이 MCP의 파일과 충돌하지 않도록 자체 `agent/` 하위 디렉터리에서 실행합니다.

각 호출에서 에이전트 내부는 다음 작업을 수행합니다. 아키텍처 섹션의 시스템 흐름 3~5단계를 확대한 내용입니다.

- inbound 요청의 `Authorization: Bearer …` header로 사용자 JWT를 받습니다.
- `BedrockAgentCoreContext`에서 workload access token을 읽습니다. AgentCore Runtime은 inbound JWT에서 이 토큰을 자동으로 파생해 에이전트에서 사용할 수 있게 합니다. 이 값은 OBO exchange의 "user subject" 입력입니다.
- workload token과 필요한 Graph scope로 AgentCore Identity의 `GetResourceOauth2Token(oauth2Flow=ON_BEHALF_OF_TOKEN_EXCHANGE)`을 호출합니다. AgentCore Identity는 Entra와 OBO 호출을 중개하고 Graph scope의 delegation token을 반환합니다.
- MCP transport 권한 부여용 M2M token을 가져옵니다(`auth_flow='M2M'`인 `@requires_access_token` decorator 사용).
- 두 HTTP header와 함께 MCP server를 호출합니다. M2M token은 `Authorization`에, Graph OBO token은 `X-Amzn-Bedrock-AgentCore-Runtime-Custom-Graph-Token`에 담습니다. 두 header는 모두 MCP Runtime의 allowlist에 포함되어 있습니다(4단계). 그렇지 않으면 AgentCore가 컨테이너에 전달하기 전에 제거합니다. 도구 인자에는 인증 데이터가 없으며 LLM은 어떤 토큰도 보지 못합니다.

> **`@requires_access_token` 참고:** Bedrock AgentCore SDK decorator에는 아직 `auth_flow="OBO"` mode가 없으므로 boto3를 통해 `GetResourceOauth2Token`을 직접 호출합니다. [OBO 개발자 가이드](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/on-behalf-of-token-exchange.html)를 참조하세요.


In [ ]:
# 에이전트가 호출할 MCP URL 계산
escaped_mcp_arn = urllib.parse.quote(mcp_launch.agent_arn, safe="")
mcp_url = f"https://bedrock-agentcore.{region}.amazonaws.com/runtimes/{escaped_mcp_arn}/invocations?qualifier=DEFAULT"
print(f"MCP URL: {mcp_url}")

In [ ]:
%%writefile agent/agent_obo.py
"""MCP 도구를 통해 Microsoft Graph를 호출하도록 Entra ID OBO 토큰 교환을 수행하는 에이전트입니다."""
import os
import boto3
from strands import Agent
from strands.models import BedrockModel
from strands.tools.mcp import MCPClient
from mcp.client.streamable_http import streamablehttp_client
from bedrock_agentcore.runtime import BedrockAgentCoreApp, BedrockAgentCoreContext
from bedrock_agentcore.identity.auth import requires_access_token

app = BedrockAgentCoreApp()

MCP_URL = os.environ['MCP_URL']
MCP_CLIENT_ID = os.environ['ENTRA_MCP_CLIENT_ID']
CREDENTIAL_PROVIDER_NAME = os.environ.get('CREDENTIAL_PROVIDER_NAME', 'entra-agent-provider')
REGION = os.environ.get('AWS_REGION', 'us-west-2')

# MCP Runtime의 allowlist에 등록된 header 이름과 일치해야 함(4단계).
# AgentCore Runtime은 이름이 'X-Amzn-Bedrock-AgentCore-Runtime-Custom-'으로 시작하고
# Runtime의 requestHeaderAllowlist에 포함된 header만 전달함.
GRAPH_TOKEN_HEADER = 'X-Amzn-Bedrock-AgentCore-Runtime-Custom-Graph-Token'

# OBO exchange에서 요청하는 Graph scope. Entra의 Agent app에 부여되고
# admin consent된 delegated permission이어야 함.
GRAPH_SCOPES = ['User.Read']

bedrock_model = BedrockModel(
    model_id='us.anthropic.claude-sonnet-4-5-20250929-v1:0',
    temperature=0.1,
)


def exchange_for_graph_token() -> str:
    """인바운드 사용자 JWT를 Graph 범위 OBO 위임 토큰으로 교환합니다.

    Runtime이 BedrockAgentCoreContext에 설정한 워크로드 액세스 토큰을 사용한 다음,
    oauth2Flow=ON_BEHALF_OF_TOKEN_EXCHANGE로 GetResourceOauth2Token을 호출합니다.
    반환된 토큰은 aud=Microsoft Graph이고 sub=user를 포함합니다. Microsoft는
    동작하는 중간 계층 서비스를 xms_act.sub 클레임에 기록합니다.

    참고: https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/on-behalf-of-token-exchange.html
    """
    workload_token = BedrockAgentCoreContext.get_workload_access_token()
    if not workload_token:
        raise RuntimeError("No workload access token on context; did Runtime deliver it?")

    client = boto3.client('bedrock-agentcore', region_name=REGION)
    response = client.get_resource_oauth2_token(
        workloadIdentityToken=workload_token,
        resourceCredentialProviderName=CREDENTIAL_PROVIDER_NAME,
        scopes=GRAPH_SCOPES,
        oauth2Flow='ON_BEHALF_OF_TOKEN_EXCHANGE',
    )
    return response['accessToken']


# @requires_access_token decorator가 MCP transport용 M2M token(client_credentials grant)을 가져옴.
# 두 흐름 모두 동일한 credential provider를 사용함.
@requires_access_token(
    provider_name='entra-agent-provider',
    scopes=[f'api://{MCP_CLIENT_ID}/.default'],
    auth_flow='M2M',
    into='m2m_token',
)
def invoke_agent(prompt: str, *, m2m_token: str) -> str:
    """두 토큰으로 invoke_agent를 실행합니다.
    - Authorization 헤더의 M2M 토큰(에이전트를 식별하고 전송에 권한을 부여함)
    - 사용자 지정 요청 헤더의 Graph OBO 토큰(MCP 도구가 Microsoft Graph를 호출할 때
      Bearer 자격 증명으로 사용함)

    LLM에는 어느 토큰도 노출되지 않습니다. 도구 시그니처에도 인증 매개변수가 없습니다.
    """
    # 1. OBO exchange: 사용자 JWT -> Graph scope의 delegation token.
    graph_token = exchange_for_graph_token()

    # 2. 두 토큰 모두 HTTP header로 전달되며 LLM context에는 도달하지 않음.
    headers = {
        'authorization': f'Bearer {m2m_token}',
        GRAPH_TOKEN_HEADER: graph_token,
    }
    mcp_client = MCPClient(lambda: streamablehttp_client(MCP_URL, headers))

    with mcp_client:
        tools = mcp_client.list_tools_sync()
        agent = Agent(
            model=bedrock_model,
            tools=tools,
            system_prompt="You are a helpful assistant. Use the available tools to answer user questions.",
        )
        return str(agent(prompt))


@app.entrypoint
def handler(payload, context):
    prompt = payload.get('prompt', 'hello')
    return invoke_agent(prompt)


if __name__ == '__main__':
    app.run()


## 7단계: AgentCore Runtime에 에이전트 배포

MCP 배포와 동일한 pattern을 사용합니다. 구성 및 시작 전에 `os.chdir('agent')`로 이동한 다음 원래 디렉터리로 돌아옵니다.

에이전트의 authorizer는 audience가 Agent app의 client ID인 JWT를 허용합니다. 앱이 자신을 위한 토큰을 요청할 때 Entra에서 bare-GUID 형식을 요구하므로 로그인 시 `api://<AGENT_CLIENT_ID>/.default`가 아닌 bare-GUID scope `<AGENT_CLIENT_ID>/.default`를 사용합니다. 그렇지 않으면 `AADSTS90009` 오류가 발생합니다. 결과 토큰의 값은 `aud = <AGENT_CLIENT_ID>`이며 authorizer가 이 값과 일치시킵니다. 전체 `AADSTS*` 오류와 해결 방법은 [ENTRA_SETUP.md](./ENTRA_SETUP.md)를 참조하세요.


In [ ]:
agent_runtime = Runtime()
agent_discovery_url = (
    f"https://login.microsoftonline.com/{os.environ['ENTRA_TENANT_ID']}/.well-known/openid-configuration"
)

try:
    os.chdir("agent")
    agent_config = agent_runtime.configure(
        entrypoint="agent_obo.py",
        auto_create_execution_role=True,
        auto_create_ecr=True,
        requirements_file="requirements.txt",
        region=region,
        agent_name="entra_obo_agent",
        authorizer_configuration={
            "customJWTAuthorizer": {
                "discoveryUrl": agent_discovery_url,
                "allowedAudience": [os.environ["ENTRA_AGENT_CLIENT_ID"]],
            }
        },
    )
finally:
    os.chdir(NOTEBOOK_DIR)
print("Agent configured")

In [ ]:
try:
    os.chdir("agent")
    agent_launch = agent_runtime.launch(  # CodeBuild 사용(로컬 Docker 불필요). Docker/Finch/Podman이 있으면 local_build=True 추가
        env_vars={
            "MCP_URL": mcp_url,
            "ENTRA_MCP_CLIENT_ID": os.environ["ENTRA_MCP_CLIENT_ID"],
            "CREDENTIAL_PROVIDER_NAME": CREDENTIAL_PROVIDER_NAME,
        }
    )
finally:
    os.chdir(NOTEBOOK_DIR)
print(f"Agent deployed: {agent_launch.agent_id}")

## 8단계: MSAL Device Code Flow로 사용자 토큰 가져오기

OBO exchange에서 대신 작업할 실제 사용자 JWT를 얻도록 실제 Entra 사용자로 로그인합니다.

로그인에서는 Agent app의 bare-GUID `.default` scope를 사용합니다. Entra는 `.default`를 Agent app에 부여된 모든 권한(여기서는 자체 `user_delegation`과 Microsoft Graph의 `User.Read`)으로 해석하고 이를 모두 포함하는 단일 동의 prompt를 표시합니다. 설정 중 admin consent를 완료했으므로 대부분의 사용자에게는 prompt가 전혀 표시되지 않습니다. 이후 OBO exchange는 이 동의를 자동으로 재사용합니다.

이 구성을 위한 포털 단계와 로그인 실패 시 사용할 문서 끝의 문제 해결 표는 [ENTRA_SETUP.md](./ENTRA_SETUP.md)를 참조하세요.


In [ ]:
import msal
import webbrowser

authority = f"https://login.microsoftonline.com/{os.environ['ENTRA_TENANT_ID']}"
scopes = [f"{os.environ['ENTRA_AGENT_CLIENT_ID']}/.default"]

msal_app = msal.PublicClientApplication(
    client_id=os.environ["ENTRA_AGENT_CLIENT_ID"],
    authority=authority,
)

result = msal_app.acquire_token_silent(scopes, account=None)

if not result:
    flow = msal_app.initiate_device_flow(scopes=scopes)
    if "error" in flow:
        print(f"Error: {flow.get('error_description')}")
    else:
        print("=" * 50)
        print(f"Go to: {flow['verification_uri']}")
        print(f"Enter code: {flow['user_code']}")
        print("=" * 50)
        webbrowser.open(flow["verification_uri"])
        result = msal_app.acquire_token_by_device_flow(flow)

if result and "access_token" in result:
    bearer_token = result["access_token"]
    print(f"\nBearer Token Received: {bearer_token[:50]}...")
else:
    print(f"Error: {result.get('error_description') if result else 'No result'}")

## 9단계: 에이전트 호출


In [ ]:
import requests
import json

escaped_agent_arn = urllib.parse.quote(agent_launch.agent_arn, safe="")
agent_url = (
    f"https://bedrock-agentcore.{region}.amazonaws.com/runtimes/{escaped_agent_arn}/invocations?qualifier=DEFAULT"
)

session_id = str(uuid.uuid4())
headers = {
    "Authorization": f"Bearer {bearer_token}",
    "Content-Type": "application/json",
    "X-Amzn-Bedrock-AgentCore-Runtime-Session-Id": session_id,
}

print(f"Invoking agent at: {agent_url[:80]}...")
response = requests.post(
    agent_url,
    data=json.dumps({"prompt": "What's my display name and email address?"}),
    headers=headers,
)
print(f"Status: {response.status_code}")
print(response.text)

## 정리


In [ ]:
agentcore_control.delete_agent_runtime(agentRuntimeId=agent_launch.agent_id)
print(f"Deleted agent: {agent_launch.agent_id}")

agentcore_control.delete_agent_runtime(agentRuntimeId=mcp_launch.agent_id)
print(f"Deleted MCP server: {mcp_launch.agent_id}")

agentcore_control.delete_oauth2_credential_provider(name=CREDENTIAL_PROVIDER_NAME)
print(f"Deleted credential provider: {CREDENTIAL_PROVIDER_NAME}")

## 보안 고려 사항

이 샘플에서는 세 가지 토큰이 이동합니다. Graph OBO token만 agent → MCP 경계를 넘으며, audience가 Microsoft Graph로 제한되고 `User.Read` scope가 적용됩니다.

| 토큰 | `aud` | 이동 경로 |
|---|---|---|
| User JWT | Agent app | user → agent `Authorization` header |
| M2M token | MCP Server app | agent → MCP `Authorization` header |
| Graph OBO token | Microsoft Graph | agent → MCP `X-Amzn-Bedrock-AgentCore-Runtime-Custom-Graph-Token` header, 이후 MCP → Graph `Authorization` header |

어떤 토큰도 prompt, 도구 signature, 도구 인자 또는 도구 반환 값에 나타나지 않습니다. MCP 도구는 request context에서 Graph token을 직접 읽습니다. 이는 delegation token을 의도한 audience로만 보내야 한다는 [Microsoft OBO 지침](https://learn.microsoft.com/en-us/entra/identity-platform/v2-oauth2-on-behalf-of-flow)을 따릅니다.

### 각 hop의 역할 분리

```
Agent → MCP Server:
    Authorization: Bearer <M2M token>                          # 에이전트 인증
    X-Amzn-Bedrock-AgentCore-Runtime-Custom-Graph-Token: <OBO token> # 사용자 delegation 전달

MCP Server → Graph API:
    Authorization: Bearer <OBO token>                          # 사용자로 작업
```

MCP server는 M2M token을 검증해 "이 요청자를 신뢰할 수 있는가"를 판단한 다음 custom header에서 OBO token을 추출하여 Graph로 전달합니다. 사용자 delegation(`sub = user`, `xms_act.sub = agent`)은 OBO token으로만 이동합니다.


## 마무리

AgentCore Identity를 통해 Microsoft Entra ID On-Behalf-Of token exchange를 수행하고 별도의 AgentCore Runtime MCP server에 있는 도구에서 Microsoft Graph를 호출하는 에이전트를 배포했습니다. 사용자 JWT는 에이전트에만 유지되었습니다. agent-to-MCP 경계를 넘은 것은 사용자 신원을 전달하고 에이전트를 acting service로 기록하는 Graph scope의 delegation token입니다.
